# This notebook shows the PySpark code that was executed on Databricks
### Due to limitations of Databricks' Free Tier, this code processed one quarter of the final subset at a time

In [ ]:
from pyspark.sql import Window
from pyspark.sql.functions import col, lit, input_file_name, regexp_extract, countDistinct, expr, lag, coalesce, when, broadcast, min, cast

In [ ]:
# Scan the entire MIT Supercloud dataset to get a "Master Map" of all CPU file paths
cpu_logs_master = (spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.csv")
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
    .load("s3a://mit-supercloud-dataset/datacenter-challenge/202201/cpu/")
    .select("path"))

# Save this so you never have to scan S3 again
cpu_logs_master.write.mode("overwrite").saveAsTable("cpu_file_inventory")

In [ ]:
# Scan the entire MIT Supercloud dataset to get a "Master Map" of all GPU file paths
gpu_logs_master = (spark.read.format("binaryFile")
    .option("recursiveFileLookup", "true")
    .option("pathGlobFilter", "*.csv")
    .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
    .load("s3a://mit-supercloud-dataset/datacenter-challenge/202201/gpu/")
    .select("path"))

# Save this so you never have to scan S3 again
gpu_logs_master.write.mode("overwrite").saveAsTable("gpu_file_inventory")

In [ ]:
# Load the Slurm Log
final_slurm_log = spark.table("final_slurm_log")

# Sanity check
test_job = final_slurm_log.filter(final_slurm_log.id_job == 4391237494359)
display(test_job)

id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
4391237494359,16618712154521,4294967294,5042570016904,61026541062099,1,['r8937440-n43543'],20,0,0,null,0,0,\N,8,9223372036854784308,normal,10003,3,525600,1623428400,1623428400,1623532270,1623575066,0,0,"1=20,2=170000,4=1,5=20,1002=1","1=20,2=170000,4=1,5=20,1002=1",OTHER,1,20,true,42796,11.887777777777778


In [ ]:
# Subset of Slurm Log for the first quarter of the final job list

spark.sql("""
CREATE OR REPLACE TABLE slurm_log_first_quarter AS
SELECT 
    s.*
FROM final_slurm_log s
INNER JOIN final_joblist_q_1 i 
    ON s.id_job = i.id_job
""")

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

In [ ]:
display(spark.table("slurm_log_first_quarter").limit(5))

id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours
26940411750496,16618712154521,4294967294,78105663882022,61026541062099,1,['r9535192-n386398'],1,0,33280,null,0,0,xeon-g6,2,9223372036854784308,normal,110692,5,1440,1609808481,1609808481,1609808481,1609818623,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:INTERACTIVE,0,1,false,10142,2.817222222222222
37012965971035,16618712154521,4294967294,20010471345300,61026541062099,1,['r6760045-n43543'],4,0,0,null,0,0,xeon-g6,4,9223372036854784308,normal,110922,6,720,1609810513,1609810513,1609810513,1609853729,0,0,"1=4,2=34000,4=1,5=4","1=4,2=34000,4=1,5=4",OTHER,0,4,false,43216,12.004444444444445
89340275724375,16618712154521,4294967294,91741573397365,61026541062099,1,['r5573787-n851693'],1,0,0,null,0,0,xeon-g6,4,9223372036854784308,normal,110367,6,720,1609818187,1609818187,1609818187,1609861410,0,0,"1=40,2=340000,4=1,5=40","1=1,2=8500,4=1,5=1",OTHER,0,40,false,43223,12.006388888888889
58578598370674,16618712154521,4294967294,60609974236499,61026541062099,1,['r1457839-n386398'],4,0,0,null,0,0,xeon-g6,8,9223372036854784308,normal,119998,6,720,1609845960,1609845960,1609845964,1609889190,0,0,"1=4,2=34000,4=1,5=4","1=4,2=34000,4=1,5=4",OTHER,0,4,false,43226,12.007222222222222
57930846613967,16618712154521,4294967294,48491787727508,61026541062099,1,['r5189505-n386398'],1,0,0,null,0,0,xeon-g6,2,9223372036854784308,normal,110009,6,1440,1609854405,1609854405,1609854405,1609940822,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:INTERACTIVE,0,1,false,86417,24.004722222222224


In [ ]:
# Join the job list to the Master Map to get the list of filenames for CPU data

# Assign tables to variables first
inventory_df = spark.table("cpu_file_inventory")
jobs_df = spark.table("final_joblist_q_1")

# Extract the ID and filter (using the variable)
inventory_with_ids = inventory_df \
    .filter(col("path").contains("timeseries")) \
    .withColumn("extracted_id", regexp_extract(col("path"), r'/(\d+)-timeseries', 1))

# Perform the join using the variables
paths_to_load = inventory_with_ids.join(
    broadcast(jobs_df), 
    inventory_with_ids["extracted_id"] == jobs_df["id_job"],
    "inner"
).select(inventory_with_ids["path"])

# Create path list
list_of_cpu_paths = [row.path for row in paths_to_load.collect()]

# Verify the count
print(f"I found {len(list_of_cpu_paths)} CPU timeseries files.")

# Sanity check for the first 5 filepaths
for path in list_of_cpu_paths[:5]:
    print(path)

I found 7649 CPU timeseries files.
s3a://mit-supercloud-dataset/datacenter-challenge/202201/cpu/0099/57543302058107-timeseries.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/cpu/0099/58026024768123-timeseries.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/cpu/0099/59349501403264-timeseries.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/cpu/0099/61645597273199-timeseries.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/cpu/0099/61984679728003-timeseries.csv


In [ ]:
# Join the job list to the Master Map to get the list of filenames for GPU data
# Same process as the CPU files, just need to change the regex

# Assign tables to variables first
inventory_df = spark.table("gpu_file_inventory")
jobs_df = spark.table("final_joblist_q_1")

# Extract the ID and filter (using the variable)
inventory_with_ids = inventory_df \
    .withColumn("extracted_id", regexp_extract(col("path"), r'/(\d+)-', 1)) \
    .filter(col("extracted_id") != "") 
    
# Perform the join using the variables
paths_to_load = inventory_with_ids.join(
    broadcast(jobs_df), 
    inventory_with_ids["extracted_id"] == jobs_df["id_job"],
    "inner"
).select(inventory_with_ids["path"])

# Create path list
list_of_gpu_paths = [row.path for row in paths_to_load.collect()]

# Verify the count
print(f"I found {len(list_of_gpu_paths)} GPU timeseries files.")

# Sanity Check
for path in list_of_gpu_paths[:5]:
    print(path)

I found 2665 GPU timeseries files.
s3a://mit-supercloud-dataset/datacenter-challenge/202201/gpu/0099/9900323770745-r3117156-n139058.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/gpu/0099/85856041358514-r5189505-n386398.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/gpu/0099/83370895129428-r4858666-n386398.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/gpu/0099/78420576576253-r2652301-n976057.csv
s3a://mit-supercloud-dataset/datacenter-challenge/202201/gpu/0099/79241358702774-r2652301-n976057.csv


In [ ]:
# Create a Master CPU Timeseries Cleaned Table
# This will contain all the data from all the CPU timeseries files
# Can distinguish between jobs using an id_job column

# Filter the CPU path list
cpu_timeseries_paths = [
    p for p in list_of_cpu_paths
    if "timeseries" in p and "summary" not in p
]
print(f"Original paths: {len(list_of_cpu_paths)}")
print(f"TimeSeries paths: {len(cpu_timeseries_paths)}")

# Read all CSV's into one DataFrame
raw_cpu_df = (spark.read.format("csv")
              .option("header", "true")
              .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
              .load(cpu_timeseries_paths)
              # This adds the file path so we can extract the Job ID
              .select("*", "_metadata.file_path"))
print(raw_cpu_df.count())

# Extract the Job ID and clean the variable types for later use
master_cpu_df = (raw_cpu_df
    .withColumn("id_job", regexp_extract(col("file_path"), r"/(\d+)-timeseries", 1))
    .withColumn("Step", col("Step").cast("string"))
    .withColumn("ElapsedTime", col("ElapsedTime").cast("double"))
    .withColumn("RSS", col("RSS").cast("double"))
    .drop("file_path")
)

Original paths: 7649
TimeSeries paths: 7649
116965602


In [ ]:
# Create a Master GPU Timeseries Cleaned Table
# This will contain the data for all GPU timeseries files
# Can distinguish between jobs using an id_job column


# Filter the GPU path list
# GPU files don't always have "timeseries" in the name. 
gpu_timeseries_paths = [
    p for p in list_of_gpu_paths 
    if "summary" not in p
]
print(f"Total GPU files found: {len(gpu_timeseries_paths)}")

# Read the CSV's into one DataFrame
raw_gpu_df = (spark.read.format("csv")
              .option("header", "true")
              .option("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.AnonymousAWSCredentialsProvider")
              .load(gpu_timeseries_paths)
              .select("*", "_metadata.file_path"))

# Add Job ID as a new column to be able to use the new DataFrame later 
master_gpu_df = (raw_gpu_df
    .withColumn("id_job", regexp_extract(col("file_path"), r"/(\d+)-", 1))
    .drop("file_path")
)

Total GPU files found: 2665


In [ ]:
# Adding a column for Elapsed Seconds to the GPU timeseries DataFrame

job_window = Window.partitionBy("id_job")

# Calculate elapsed_seconds
master_gpu_df = (master_gpu_df
    .withColumn("start_time", min(col("timestamp")).over(job_window))
    .withColumn("elapsed_seconds", 
                col("timestamp").cast("double").cast("long") - 
                col("start_time").cast("double").cast("long"))
    .drop("start_time")
)

# Sanity Check
display(
    master_gpu_df.select("id_job", "timestamp", "elapsed_seconds")
    .sort("id_job", "elapsed_seconds")
    .limit(5)
)

id_job,timestamp,elapsed_seconds
10013613296509,1622053509.265,0
10013613296509,1622053509.369,0
10013613296509,1622053509.577,0
10013613296509,1622053509.472,0
10013613296509,1622053509.682,0


In [ ]:
# Calculate the number of steps for each job and put it in the slurm log subset

step_counts_df = (master_cpu_df
                  .groupBy("id_job")
                  .agg(countDistinct("Step").alias("Job_Steps")))
print(f"Aggregated steps for {step_counts_df.count()} jobs.")

slurm_subset = spark.table("slurm_log_first_quarter")
updated_slurm_subset = slurm_subset.join(step_counts_df, on="id_job", how="left")

# Save the updated table
updated_slurm_subset.write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("slurm_log_first_quarter")

Aggregated steps for 7649 jobs.


In [ ]:
# Creating CPU Utilization features

# Aggregate all metrics in one pass
# This collapses the millions of telemetry rows into a single summary row per job
cpu_stats_df = (master_cpu_df
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(
        expr("MAX(try_cast(CPUUtilization as double))").alias("Max0"),
        expr("MIN(try_cast(CPUUtilization as double))").alias("Min0"),
        expr("percentile(try_cast(CPUUtilization as double), 0.5)").alias("Med0"),
        expr("percentile(try_cast(CPUUtilization as double), 0.75) - percentile(try_cast(CPUUtilization as double), 0.25)").alias("IQR0")
    )
)

# Dividing by the number of CPU's allocated to normalize the utilization percentages
slurm_base = spark.table("slurm_log_first_quarter").drop("Max_CPU_Util", "Med_CPU_Util", "IQR_CPU_Util")
final_slurm = (slurm_base.join(cpu_stats_df, on="id_job", how="left")
    .withColumn("Max_CPU_Util", col("Max0") / col("cpus_alloc").cast("double"))
    .withColumn("Min_CPU_Util", col("Min0") / col("cpus_alloc").cast("double"))
    .withColumn("Med_CPU_Util", col("Med0") / col("cpus_alloc").cast("double"))
    .withColumn("IQR_CPU_Util", col("IQR0") / col("cpus_alloc").cast("double"))
    .drop("Max0", "Min0", "Med0", "IQR0")
)

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.select("id_job", "cpus_alloc", "Max_CPU_Util", "Min_CPU_Util", "Med_CPU_Util", "IQR_CPU_Util")
        .limit(10)
)

id_job,cpus_alloc,Max_CPU_Util,Min_CPU_Util,Med_CPU_Util,IQR_CPU_Util
44577838772074,1,390.0,1.1,100.0,0.09999999999999432
38371777066344,1,1934.9,987.4,1547.0,89.02499999999986
82235524800475,1,642.2,267.3,498.95,74.02500000000009
3951762038676,1,2872.1,300.7,2394.05,548.7750000000001
18615531464887,4,1000.275,0.0,0.0,0.025
57188562569952,1,3070.0,0.0,5.0,7.025000000000001
32969260300898,40,8.74,0.0,0.0,0.0
45263363846346,1,440.0,0.0,99.7,2.0999999999999943
25428864798032,1,314530.0,76.2,899.245,159.04999999999995
63204698117087,4,115.0,0.0,0.0,0.05


In [ ]:
# Applying same logic as above for: RSS, VM Size, Pages, ReadMB, and WriteMB features

target_metrics = ["RSS", "VMSize", "Pages", "ReadMB", "WriteMB"]

# Build the aggregate expressions dynamically
agg_expressions = []
for m in target_metrics:
    agg_expressions.append(expr(f"MAX(try_cast({m} as double))").alias(f"Max_{m}"))
    agg_expressions.append(expr(f"MIN(try_cast({m} as double))").alias(f"Min_{m}"))
    agg_expressions.append(expr(f"percentile(try_cast({m} as double), 0.5)").alias(f"Med_{m}"))
    agg_expressions.append(expr(f"percentile(try_cast({m} as double), 0.75) - percentile(try_cast({m} as double), 0.25)").alias(f"IQR_{m}"))

# Process the entire master_cpu_df
# We filter once, group by ID, and run all calculations
metrics_summary_df = (master_cpu_df
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(*agg_expressions) # The '*' unpacks the list into the agg function
)

new_col_names = [f"{stat}_{m}" for m in target_metrics for stat in ["Max", "Min",  "Med", "IQR"]]
slurm_base = spark.table("slurm_log_first_quarter").drop(*new_col_names)
final_slurm = slurm_base.join(metrics_summary_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

display(final_slurm.limit(5))

id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours,Job_Steps,Max_CPU_Util,Min_CPU_Util,Med_CPU_Util,IQR_CPU_Util,Max_RSS,Min_RSS,Med_RSS,IQR_RSS,Max_VMSize,Min_VMSize,Med_VMSize,IQR_VMSize,Max_Pages,Min_Pages,Med_Pages,IQR_Pages,Max_ReadMB,Min_ReadMB,Med_ReadMB,IQR_ReadMB,Max_WriteMB,Min_WriteMB,Med_WriteMB,IQR_WriteMB
44577838772074,12603536456035,107,36880203429568,61026541062099,1,['r9040233-n386398'],1,0,253,null,0,0,xeon-g6,8,9223372036854784308,normal,10014,11,525600,1612391473,1612391474,1612398284,1612423969,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:BATCH,0,1,false,25685,7.134722222222222,2,390.0,1.1,100.0,0.09999999999999432,8.6890452E7,226204.0,343932.0,34844.0,1.54591168E8,283556.0,464412.0,35612.0,12.0,4.0,4.0,0.0,65.197869,0.0,0.0,0.0,6.346914,0.0,0.0,0.0
38371777066344,105186008840,102,30273376745109,61026541062099,1,['r8607415-n976057'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473846,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14414,4.003888888888889,2,1934.9,987.4,1547.0,89.02499999999986,1.1412224E7,735440.0,1254032.0,158138.0,1.86215632E8,1.9851532E7,2.95303E7,165760.0,1432.0,116.0,773.5,601.75,9.072832,1.371466,6.053281,0.20069625000000002,15.274106,0.045257,0.08653050000000001,0.014340499999999992
82235524800475,105186008840,176,30273376745109,61026541062099,1,['r9535192-n911952'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473850,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14418,4.005,2,642.2,267.3,498.95,74.02500000000009,1.014266E7,677472.0,1166956.0,96239.0,1.8545998E8,1.9819472E7,2.9480202E7,106360.0,516.0,27.0,295.5,247.0,3.23868,0.006182,2.642689,2.9888584999999996,0.050894999999999996,0.007927,0.0292585,0.011533749999999999
3951762038676,89475896797385,455,30273376745109,61026541062099,1,['r2825489-n139058'],1,0,32512,null,0,0,xeon-g6,4,15360,normal,10300,5,360,1612494659,1612494659,1612516270,1612536435,0,0,"1=1,2=15360,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,20165,5.601388888888889,2,2872.1,300.7,2394.05,548.7750000000001,1.1678892E7,832444.0,1424544.0,69732.0,1.86510924E8,1.9474024E7,2.9715966E7,78455.0,12410.0,131.0,7853.0,4450.75,378.228218,4.8600000000000005E-4,6.0652355,0.6014877500000004,27.027093,0.001086,0.095277,0.028831750000000003
18615531464887,16618712154521,4294967294,11051103666595,61026541062099,1,['r5189505-n685852'],4,0,0,null,0,0,xeon-g6,4,9223372036854784308,normal,110473,6,720,1617492816,1617492816,1617492817,1617536031,0,0,"1=4,2=34000,4=1,5=4,1002=1","1=4,2=34000,4=1,5=4,1002=1",OTHER,1,4,false,43214,12.00388888888889,3,1000.275,0.0,0.0,0.025,3492364.0,9912.0,58772.0,2351784.0,1.8636848E7,319660.0,1493938.0,1.6973192E7,549.0,6.0,32.5,520.0,492.406921,0.0,0.0,0.0,2.988339,0.0,0.0,0.0


In [ ]:
# Adding RSS Velocity to CPU Timeseries dataframe 

job_window = Window.partitionBy("id_job").orderBy(col("ElapsedTime").cast("double"))

master_velocity_df = (master_cpu_df
    .filter(col("Step").isin("0", "batch"))
    .withColumn("RSS_Velocity", 
        coalesce(
            col("RSS").cast("double") - lag(col("RSS").cast("double"), 1).over(job_window), 
            lit(0)
        )
    )
)

# Sanity Check
display(master_velocity_df.limit(5))

Step,Node,Series,ElapsedTime,EpochTime,CPUFrequency,CPUTime,CPUUtilization,RSS,VMSize,Pages,ReadMB,WriteMB,id_job,RSS_Velocity
batch,r5189505-n851693,0,0.0,1612459432,3276,0.0,0.0,2960.0,199048,0,0.0,0.0,10240855146456,0.0
batch,r5189505-n851693,0,10.0,1612459442,3276,0.4,4.0,22772.0,62684,5,38.567101,0.077137,10240855146456,19812.0
batch,r5189505-n851693,0,20.0,1612459452,3276,3.57,35.7,45764.0,2731176,17,4.670118,7.000000000000001e-06,10240855146456,22992.0
batch,r5189505-n851693,0,30.0,1612459462,3276,0.42,4.2,61880.0,2750068,19,3.7881120000000004,0.0,10240855146456,16116.0
batch,r5189505-n851693,0,40.0,1612459472,3276,0.51,5.1,121336.0,2999368,82,3.111468,0.0,10240855146456,59456.0


In [ ]:
# Creating RSS Velocity features

target_metrics = ["RSS_Velocity"]

# Build the aggregate expressions dynamically
agg_expressions = []
for m in target_metrics:
    agg_expressions.append(expr(f"MAX(try_cast({m} as double))").alias(f"Max_{m}"))
    agg_expressions.append(expr(f"MIN(try_cast({m} as double))").alias(f"Min_{m}"))
    agg_expressions.append(expr(f"percentile(try_cast({m} as double), 0.5)").alias(f"Med_{m}"))
    agg_expressions.append(expr(f"percentile(try_cast({m} as double), 0.75) - percentile(try_cast({m} as double), 0.25)").alias(f"IQR_{m}"))

# Process the entire master_cpu_df
metrics_summary_df = (master_velocity_df
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(*agg_expressions)
)

new_col_names = [f"{stat}_{m}" for m in target_metrics for stat in ["Max", "Min", "Med", "IQR"]]
slurm_base = spark.table("slurm_log_first_quarter").drop(*new_col_names)
final_slurm = slurm_base.join(metrics_summary_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.limit(5))

id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours,Job_Steps,Max_CPU_Util,Min_CPU_Util,Med_CPU_Util,IQR_CPU_Util,Max_RSS,Min_RSS,Med_RSS,IQR_RSS,Max_VMSize,Min_VMSize,Med_VMSize,IQR_VMSize,Max_Pages,Min_Pages,Med_Pages,IQR_Pages,Max_ReadMB,Min_ReadMB,Med_ReadMB,IQR_ReadMB,Max_WriteMB,Min_WriteMB,Med_WriteMB,IQR_WriteMB,Max_RSS_Velocity,Min_RSS_Velocity,Med_RSS_Velocity,IQR_RSS_Velocity
44577838772074,12603536456035,107,36880203429568,61026541062099,1,['r9040233-n386398'],1,0,253,null,0,0,xeon-g6,8,9223372036854784308,normal,10014,11,525600,1612391473,1612391474,1612398284,1612423969,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:BATCH,0,1,false,25685,7.134722222222222,2,390.0,1.1,100.0,0.09999999999999432,8.6890452E7,226204.0,343932.0,34844.0,1.54591168E8,283556.0,464412.0,35612.0,12.0,4.0,4.0,0.0,65.197869,0.0,0.0,0.0,6.346914,0.0,0.0,0.0,2.364212E7,-8.6604148E7,0.0,0.0
38371777066344,105186008840,102,30273376745109,61026541062099,1,['r8607415-n976057'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473846,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14414,4.003888888888889,2,1934.9,987.4,1547.0,89.02499999999986,1.1412224E7,735440.0,1254032.0,158138.0,1.86215632E8,1.9851532E7,2.95303E7,165760.0,1432.0,116.0,773.5,601.75,9.072832,1.371466,6.053281,0.20069625000000002,15.274106,0.045257,0.08653050000000001,0.014340499999999992,1.0077004E7,-1.0218668E7,480.0,182112.0
82235524800475,105186008840,176,30273376745109,61026541062099,1,['r9535192-n911952'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473850,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14418,4.005,2,642.2,267.3,498.95,74.02500000000009,1.014266E7,677472.0,1166956.0,96239.0,1.8545998E8,1.9819472E7,2.9480202E7,106360.0,516.0,27.0,295.5,247.0,3.23868,0.006182,2.642689,2.9888584999999996,0.050894999999999996,0.007927,0.0292585,0.011533749999999999,8983872.0,-8971496.0,1544.0,156011.0
3951762038676,89475896797385,455,30273376745109,61026541062099,1,['r2825489-n139058'],1,0,32512,null,0,0,xeon-g6,4,15360,normal,10300,5,360,1612494659,1612494659,1612516270,1612536435,0,0,"1=1,2=15360,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,20165,5.601388888888889,2,2872.1,300.7,2394.05,548.7750000000001,1.1678892E7,832444.0,1424544.0,69732.0,1.86510924E8,1.9474024E7,2.9715966E7,78455.0,12410.0,131.0,7853.0,4450.75,378.228218,4.8600000000000005E-4,6.0652355,0.6014877500000004,27.027093,0.001086,0.095277,0.028831750000000003,1.0253056E7,-1.0254788E7,202.0,84161.0
18615531464887,16618712154521,4294967294,11051103666595,61026541062099,1,['r5189505-n685852'],4,0,0,null,0,0,xeon-g6,4,9223372036854784308,normal,110473,6,720,1617492816,1617492816,1617492817,1617536031,0,0,"1=4,2=34000,4=1,5=4,1002=1","1=4,2=34000,4=1,5=4,1002=1",OTHER,1,4,false,43214,12.00388888888889,3,1000.275,0.0,0.0,0.025,3492364.0,9912.0,58772.0,2351784.0,1.8636848E7,319660.0,1493938.0,1.6973192E7,549.0,6.0,32.5,520.0,492.406921,0.0,0.0,0.0,2.988339,0.0,0.0,0.0,3482452.0,-3482452.0,0.0,4703568.0


In [ ]:
# RSS Slope and RSS Acceleration features

rss_trends_df = (master_cpu_df
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(
        expr("REGR_SLOPE(try_cast(RSS as double), try_cast(ElapsedTime as double))").alias("RSS_Slope"),
        expr("REGR_SLOPE(try_cast(RSS as double), POWER(try_cast(ElapsedTime as double), 2))").alias("RSS_Accel")
    )
)

slurm_base = spark.table("slurm_log_first_quarter").drop("RSS_Slope", "RSS_Accel")
final_slurm = slurm_base.join(rss_trends_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.sort(col("RSS_Slope").desc()).select("id_job", "RSS_Slope", "RSS_Accel").limit(5))

id_job,RSS_Slope,RSS_Accel
29064929193107,119349.10667187566,11.382726966245277
53242368500299,105769.97301305039,12.724984008398355
44050050979602,75355.78119476177,10.044442523538374
65986983938800,74466.75297289343,9.92957721529882
8589293801078,63615.84651966421,6.460676865819337


In [ ]:
# VMSize_Slope feature

vm_trends_df = (master_cpu_df
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(
        expr("REGR_SLOPE(try_cast(VMSize as double), try_cast(ElapsedTime as double))").alias("VMSize_Slope")
    )
)

slurm_base = spark.table("slurm_log_first_quarter").drop("VMSize_Slope")
final_slurm = slurm_base.join(vm_trends_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.sort(col("VMSize_Slope").desc()).select("id_job", "VMSize_Slope").limit(5))

id_job,VMSize_Slope
37677752901464,1247095.2775097426
54969241154228,1199582.0846106221
67153746344476,1137832.6363962325
1373475150404,1132229.1902188477
91243829040586,972864.1572036293


In [ ]:
# Creating new column in CPU Timeseries file for "VMSize - RSS Gap"

# Calculate the Gap for all jobs at once
cpu_timeseries_cleaned = master_cpu_df.withColumn(
    "VM_RSS_Gap", 
    col("VMSize").cast("double") - col("RSS").cast("double")
)

# Sanity Check
display(cpu_timeseries_cleaned.select("id_job", "VMSize", "RSS", "VM_RSS_Gap")
        .orderBy("id_job")
        .limit(10))

id_job,VMSize,RSS,VM_RSS_Gap
10013613296509,168348,2204.0,166144.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0
10013613296509,7776,824.0,6952.0


In [ ]:
# VM - RSS Gap Features

target_metrics = ["VM_RSS_Gap"]

# Build the aggregate expressions dynamically
agg_expressions = []
for m in target_metrics:
    agg_expressions.append(expr(f"MAX(try_cast({m} as double))").alias(f"Max_{m}"))
    agg_expressions.append(expr(f"MIN(try_cast({m} as double))").alias(f"Min_{m}"))
    agg_expressions.append(expr(f"percentile(try_cast({m} as double), 0.5)").alias(f"Med_{m}"))
    agg_expressions.append(expr(f"percentile(try_cast({m} as double), 0.75) - percentile(try_cast({m} as double), 0.25)").alias(f"IQR_{m}"))

# Process the entire master_cpu_df
metrics_summary_df = (cpu_timeseries_cleaned
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(*agg_expressions) # The '*' unpacks the list into the agg function
)

new_col_names = [f"{stat}_{m}" for m in target_metrics for stat in ["Max", "Min", "Med", "IQR"]]
slurm_base = spark.table("slurm_log_first_quarter").drop(*new_col_names)
final_slurm = slurm_base.join(metrics_summary_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.limit(5))

id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours,Job_Steps,Max_CPU_Util,Min_CPU_Util,Med_CPU_Util,IQR_CPU_Util,Max_RSS,Min_RSS,Med_RSS,IQR_RSS,Max_VMSize,Min_VMSize,Med_VMSize,IQR_VMSize,Max_Pages,Min_Pages,Med_Pages,IQR_Pages,Max_ReadMB,Min_ReadMB,Med_ReadMB,IQR_ReadMB,Max_WriteMB,Min_WriteMB,Med_WriteMB,IQR_WriteMB,Max_RSS_Velocity,Min_RSS_Velocity,Med_RSS_Velocity,IQR_RSS_Velocity,RSS_Slope,RSS_Accel,VMSize_Slope,Max_VM_RSS_Gap,Min_VM_RSS_Gap,Med_VM_RSS_Gap,IQR_VM_RSS_Gap
44577838772074,12603536456035,107,36880203429568,61026541062099,1,['r9040233-n386398'],1,0,253,null,0,0,xeon-g6,8,9223372036854784308,normal,10014,11,525600,1612391473,1612391474,1612398284,1612423969,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:BATCH,0,1,false,25685,7.134722222222222,2,390.0,1.1,100.0,0.09999999999999432,8.6890452E7,226204.0,343932.0,34844.0,1.54591168E8,283556.0,464412.0,35612.0,12.0,4.0,4.0,0.0,65.197869,0.0,0.0,0.0,6.346914,0.0,0.0,0.0,2.364212E7,-8.6604148E7,0.0,0.0,-1744.2918042177998,-0.22462989970831274,-2020.4038062701075,8.4832344E7,44224.0,121468.0,2204.0
38371777066344,105186008840,102,30273376745109,61026541062099,1,['r8607415-n976057'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473846,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14414,4.003888888888889,2,1934.9,987.4,1547.0,89.02499999999986,1.1412224E7,735440.0,1254032.0,158138.0,1.86215632E8,1.9851532E7,2.95303E7,165760.0,1432.0,116.0,773.5,601.75,9.072832,1.371466,6.053281,0.20069625000000002,15.274106,0.045257,0.08653050000000001,0.014340499999999992,1.0077004E7,-1.0218668E7,480.0,182112.0,27.86193147296362,0.004727829680479276,818.5660720139324,1.75289704E8,1.9086932E7,2.82874E7,33625.0
82235524800475,105186008840,176,30273376745109,61026541062099,1,['r9535192-n911952'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473850,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14418,4.005,2,642.2,267.3,498.95,74.02500000000009,1.014266E7,677472.0,1166956.0,96239.0,1.8545998E8,1.9819472E7,2.9480202E7,106360.0,516.0,27.0,295.5,247.0,3.23868,0.006182,2.642689,2.9888584999999996,0.050894999999999996,0.007927,0.0292585,0.011533749999999999,8983872.0,-8971496.0,1544.0,156011.0,22.52910840099201,0.0028999097738430317,249.29295446481902,1.75349976E8,1.9117076E7,2.832462E7,30340.0
3951762038676,89475896797385,455,30273376745109,61026541062099,1,['r2825489-n139058'],1,0,32512,null,0,0,xeon-g6,4,15360,normal,10300,5,360,1612494659,1612494659,1612516270,1612536435,0,0,"1=1,2=15360,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,20165,5.601388888888889,2,2872.1,300.7,2394.05,548.7750000000001,1.1678892E7,832444.0,1424544.0,69732.0,1.86510924E8,1.9474024E7,2.9715966E7,78455.0,12410.0,131.0,7853.0,4450.75,378.228218,4.8600000000000005E-4,6.0652355,0.6014877500000004,27.027093,0.001086,0.095277,0.028831750000000003,1.0253056E7,-1.0254788E7,202.0,84161.0,19.5046002610967,0.004184084759266744,335.9233278974562,1.75315604E8,1.8627276E7,2.829328E7,52062.0
18615531464887,16618712154521,4294967294,11051103666595,61026541062099,1,['r5189505-n685852'],4,0,0,null,0,0,xeon-g6,4,9223372036854784308,normal,110473,6,720,1617492816,1617492816,1617492817,1617536031,0,0,"1=4,2=34000,4=1,5=4,1002=1","1=4,2=34000,4=1,5=4,1002=1",OTHER,1,4,false,43214,12.00388888888889,3,1000.275,0.0,0.0,0.025,3492364.0,9912.0,58772.0,2351784.0,1.8636848E7,319660.0,1493938.0,1.6973192E7,549.0,6.0,32.5,520.0,492.406921,0.0,0.0,0.0,2.988339,0.0,0.0,0.0,3482452.0,-3482452.0,0.0,4703568.0,96.28263515512374,0.008849007790361463,686.85864574

In [ ]:
#  VM_RSS_Gap_Slope feature

# Calculate Regression Metric
vm_rss_slope_df = (cpu_timeseries_cleaned
    .filter(
        (col("Step").isin("0", "batch")) & 
        (col("ElapsedTime").cast("double") > 120) & 
        (col("ElapsedTime").cast("double") <= 7200)
    )
    .groupBy("id_job")
    .agg(
        expr("REGR_SLOPE(try_cast(VM_RSS_Gap as double), try_cast(ElapsedTime as double))").alias("VM_RSS_Gap_Slope")
    )
)

slurm_base = spark.table("slurm_log_first_quarter").drop("VM_RSS_Gap_Slope")
final_slurm = slurm_base.join(vm_rss_slope_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.sort(col("VM_RSS_Gap_Slope").desc()).select("id_job", "VM_RSS_Gap_Slope").limit(5))

id_job,VM_RSS_Gap_Slope
37677752901464,1246367.2633609942
54969241154228,1186903.7171223955
67153746344476,1126866.1487927258
1373475150404,1085005.6280299155
91243829040586,972166.8267002796


In [ ]:
# Creating GPU_Util, Mem_Util, Temp_GPU, Temp_Mem, and Power_Draw features 

# Define the GPU metrics and their new variable names
target_metrics = ["utilization_gpu_pct", "utilization_memory_pct", 
                  "temperature_gpu", "temperature_memory", "power_draw_W"]

var_names = ["GPU_Util", "Mem_Util", "Temp_GPU", "Temp_Mem", "Power_Draw"]

# Build the aggregate expressions dynamically
agg_expressions = []
for metric, name in zip(target_metrics, var_names):
    agg_expressions.append(expr(f"MAX(try_cast({metric} as double))").alias(f"Max_{name}"))
    agg_expressions.append(expr(f"MIN(try_cast({metric} as double))").alias(f"Min_{name}"))
    agg_expressions.append(expr(f"percentile(try_cast({metric} as double), 0.5)").alias(f"Med_{name}"))
    agg_expressions.append(expr(f"percentile(try_cast({metric} as double), 0.75) - "
                                f"percentile(try_cast({metric} as double), 0.25)").alias(f"IQR_{name}"))

# Filter and Aggregate for all jobs
gpu_metrics_df = (master_gpu_df
    .filter(
        (col("elapsed_seconds") > 120) & 
        (col("elapsed_seconds") <= 7200)
    )
    .groupBy("id_job")
    .agg(*agg_expressions)
)

new_cols = [f"{stat}_{name}" for name in var_names for stat in ["Max", "Min", "Med", "IQR"]]
slurm_base = spark.table("slurm_log_first_quarter").drop(*new_cols)
final_slurm = slurm_base.join(gpu_metrics_df, on="id_job", how="left")

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

# Sanity Check
display(final_slurm.select("id_job", "Max_Temp_GPU", "Max_Power_Draw", "Max_GPU_Util").limit(20))

id_job,Max_Temp_GPU,Max_Power_Draw,Max_GPU_Util
44577838772074,null,null,null
38371777066344,null,null,null
82235524800475,null,null,null
3951762038676,null,null,null
18615531464887,42.0,120.86,85.0
57188562569952,null,null,null
32969260300898,null,null,null
45263363846346,null,null,null
25428864798032,null,null,null
63204698117087,null,null,null


In [ ]:
# Power Util Ratio feature

slurm_base = spark.table("slurm_log_first_quarter")

final_slurm = slurm_base.withColumn(
    "Power_Util_Ratio",
    when(col("Med_GPU_Util") > 0, 
         col("Med_Power_Draw") / col("Med_GPU_Util"))
    .otherwise(None)
)

# Save the updated table
final_slurm.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("slurm_log_first_quarter")

In [ ]:
# Print Final Table for the first quarter of the final subset
display(spark.table("slurm_log_first_quarter").limit(5))

id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,gpus_alloc,cpus_alloc,job_completed,duration_seconds,duration_hours,Job_Steps,Max_CPU_Util,Min_CPU_Util,Med_CPU_Util,IQR_CPU_Util,Max_RSS,Min_RSS,Med_RSS,IQR_RSS,Max_VMSize,Min_VMSize,Med_VMSize,IQR_VMSize,Max_Pages,Min_Pages,Med_Pages,IQR_Pages,Max_ReadMB,Min_ReadMB,Med_ReadMB,IQR_ReadMB,Max_WriteMB,Min_WriteMB,Med_WriteMB,IQR_WriteMB,Max_RSS_Velocity,Min_RSS_Velocity,Med_RSS_Velocity,IQR_RSS_Velocity,RSS_Slope,RSS_Accel,VMSize_Slope,Max_VM_RSS_Gap,Min_VM_RSS_Gap,Med_VM_RSS_Gap,IQR_VM_RSS_Gap,VM_RSS_Gap_Slope,Max_GPU_Util,Min_GPU_Util,Med_GPU_Util,IQR_GPU_Util,Max_Mem_Util,Min_Mem_Util,Med_Mem_Util,IQR_Mem_Util,Max_Temp_GPU,Min_Temp_GPU,Med_Temp_GPU,IQR_Temp_GPU,Max_Temp_Mem,Min_Temp_Mem,Med_Temp_Mem,IQR_Temp_Mem,Max_Power_Draw,Min_Power_Draw,Med_Power_Draw,IQR_Power_Draw,Power_Util_Ratio
44577838772074,12603536456035,107,36880203429568,61026541062099,1,['r9040233-n386398'],1,0,253,null,0,0,xeon-g6,8,9223372036854784308,normal,10014,11,525600,1612391473,1612391474,1612398284,1612423969,0,0,"1=1,2=8500,4=1,5=1","1=1,2=8500,4=1,5=1",LLSUB:BATCH,0,1,false,25685,7.134722222222222,2,390.0,1.1,100.0,0.09999999999999432,8.6890452E7,226204.0,343932.0,34844.0,1.54591168E8,283556.0,464412.0,35612.0,12.0,4.0,4.0,0.0,65.197869,0.0,0.0,0.0,6.346914,0.0,0.0,0.0,2.364212E7,-8.6604148E7,0.0,0.0,-1744.2918042177998,-0.22462989970831274,-2020.4038062701075,8.4832344E7,44224.0,121468.0,2204.0,-276.11200205230733,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
38371777066344,105186008840,102,30273376745109,61026541062099,1,['r8607415-n976057'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473846,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14414,4.003888888888889,2,1934.9,987.4,1547.0,89.02499999999986,1.1412224E7,735440.0,1254032.0,158138.0,1.86215632E8,1.9851532E7,2.95303E7,165760.0,1432.0,116.0,773.5,601.75,9.072832,1.371466,6.053281,0.20069625000000002,15.274106,0.045257,0.08653050000000001,0.014340499999999992,1.0077004E7,-1.0218668E7,480.0,182112.0,27.86193147296362,0.004727829680479276,818.5660720139324,1.75289704E8,1.9086932E7,2.82874E7,33625.0,790.7041405409981,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
82235524800475,105186008840,176,30273376745109,61026541062099,1,['r9535192-n911952'],1,0,0,null,0,0,xeon-g6,4,15360,normal,10365,6,240,1612459422,1612459428,1612459432,1612473850,0,0,"1=1,2=15360,3=18446744073709551614,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,14418,4.005,2,642.2,267.3,498.95,74.02500000000009,1.014266E7,677472.0,1166956.0,96239.0,1.8545998E8,1.9819472E7,2.9480202E7,106360.0,516.0,27.0,295.5,247.0,3.23868,0.006182,2.642689,2.9888584999999996,0.050894999999999996,0.007927,0.0292585,0.011533749999999999,8983872.0,-8971496.0,1544.0,156011.0,22.52910840099201,0.0028999097738430317,249.29295446481902,1.75349976E8,1.9117076E7,2.832462E7,30340.0,226.7638460638357,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
3951762038676,89475896797385,455,30273376745109,61026541062099,1,['r2825489-n139058'],1,0,32512,null,0,0,xeon-g6,4,15360,normal,10300,5,360,1612494659,1612494659,1612516270,1612536435,0,0,"1=1,2=15360,4=1,5=1","1=1,2=15360,4=1,5=1",OTHER,0,1,false,20165,5.601388888888889,2,2872.1,300.7,2394.05,548.7750000000001,1.1678892E7,832444.0,1424544.0,69732.0,1.86510924E8,1.9474024E7,2.9715966E7,78455.0,12410.0,131.0,7853.0,4450.75,378.228218,4.8600000000000005E-4,6.0652355,0.6014877500000004,27.027093,0.001086,0.095277,0.028831750000000003,1.0253056E7,-1.0254788E7,